In [3]:
from __future__ import print_function, division
from optparse import OptionParser
from collections import defaultdict, OrderedDict
import concurrent.futures
import sys
import os
import rhalphalib as rl
import numpy as np
import pickle
import gzip
import json
from coffea import hist, processor
from coffea.util import load, save
from scipy import stats
import ROOT

rl.util.install_roofit_helpers()
rl.ParametericSample.PreferRooParametricHist = False

mT_binning_dict = {
    "2018": np.linspace(0,600,21),
    "2017": np.linspace(0,600,21),
    "2016": np.linspace(0,600,21)
}

Welcome to JupyROOT 6.18/00


In [4]:
def remap_histograms(hists):
    
    '''
    This method is used for mapping  the histograms in the .scaled file
    '''
    data_hists = {}
    bkg_hists = {}
    signal_hists = {}

    process = hist.Cat("process", "Process", sorting="placement")
    cats = ("process",)
    sig_map = OrderedDict()
    bkg_map = OrderedDict()
    data_map = OrderedDict()
    bkg_map["Hbb"] = ("Hbb*",)
    bkg_map["DY+jets"] = ("DY+*",)
    bkg_map["VV"] = (["WW", "WZ", "ZZ"],)
    bkg_map["ST"] = ("ST*",)
    bkg_map["TT"] = ("TT*",)
    bkg_map["W+jets"] = ("W+*",)
#     bkg_map["W+HF"] = ("W+HF",)
#     bkg_map["W+LF"] = ("W+LF",)
    bkg_map["Z+jets"] = ("Z+*",)
    bkg_map["QCD"] = ("QCD*",)
    data_map["SingleElectron"] = ("SingleElectron",)
    data_map["SingleMuon"] = ("SingleMuon",)
    data_map["EGamma"] = ("EGamma",)
    sig_map["Mphi-1995_Mchi-1000"] = ("*Mphi-1995_Mchi-1000*",)
    
#    processed signal 
    for signal in hists['sig']['template'].identifiers('process'):
        if 'Mphi' not in str(signal): continue
        sig_map[str(signal)] = (str(signal),)  ## signals

    for key in hists["data"].keys():
        bkg_hists[key] = hists["bkg"][key].group(cats, process, bkg_map)
        signal_hists[key] = hists["sig"][key].group(cats, process, sig_map)
        data_hists[key] = hists["data"][key].group(cats, process, data_map)

    print('initial mT binning',bkg_hists["template"].axis("mT").edges())


    bkg_hists["template"] = bkg_hists["template"].rebin("mT", hist.Bin("mT", "mT", mT_binning))
    
    signal_hists["template"] = signal_hists["template"].rebin("mT", hist.Bin("mT", "mT", mT_binning))
    data_hists["template"] = data_hists["template"].rebin("mT", hist.Bin("mT", "mT", mT_binning))

    hists = {"bkg": bkg_hists, "data": data_hists, "sig": signal_hists}

    return hists


In [5]:
def template(dictionary, process, systematic, region, read_sumw2=False):
    histogram = dictionary[region].integrate("process", process)
    nominal, sumw2 = histogram.integrate("systematic", "nominal").values(sumw2=True)[()]
    zerobins = nominal <= 0.
    output = nominal
    
    if "nominal" not in systematic and "data" not in systematic:
    #print('Normalizing',systematic,'histogram of',process,'in region',region)
        output = histogram.integrate("systematic", systematic).values()[()]
        output[zerobins] = 1.
        output[~zerobins] /= nominal[~zerobins]
        output[np.isnan(output)] = 1.
        
    binning = (
        dictionary[region]
        .integrate("process", process)
        .integrate("systematic", systematic)
        .axis("mT")
        .edges()
    )
    if read_sumw2:
        return (output, binning, "mT", sumw2)
    return (output, binning, "mT")

In [8]:
!pwd

/cms/scratch/runiyal/new_coffea/decaf/analysis/Rishabh_NB/new_coffea_nbs/Combine related


In [21]:
mT_binning = mT_binning_dict['2017']
mTbins = np.array(mT_binning)
nmT = len(mTbins) - 1
hists = load("../../study_scaled_files/no_dphi_sel_lep_monotop_v1_2017.scaled")
# hists = remap_histograms(hists)

In [22]:
hists['bkg']['template'].integrate('process').integrate('region').values()

{('nominal',): array(nan),
 ('btagUp',): array(nan),
 ('btagDown',): array(nan),
 ('qcd1Up',): array(85238.835514),
 ('qcd1Down',): array(85238.835514),
 ('qcd2Up',): array(85238.835514),
 ('qcd2Down',): array(85238.835514),
 ('qcd3Up',): array(85238.835514),
 ('qcd3Down',): array(85238.835514),
 ('muFUp',): array(85238.835514),
 ('muFDown',): array(85238.835514),
 ('muRUp',): array(85238.835514),
 ('muRDown',): array(85238.835514),
 ('ew1Up',): array(85238.835514),
 ('ew1Down',): array(85238.835514),
 ('ew2GUp',): array(85238.835514),
 ('ew2GDown',): array(85238.835514),
 ('ew2WUp',): array(85238.835514),
 ('ew2WDown',): array(85238.835514),
 ('ew2ZUp',): array(85238.835514),
 ('ew2ZDown',): array(85238.835514),
 ('ew3GUp',): array(85238.835514),
 ('ew3GDown',): array(85238.835514),
 ('ew3WUp',): array(85238.835514),
 ('ew3WDown',): array(85238.835514),
 ('ew3ZUp',): array(85238.835514),
 ('ew3ZDown',): array(85238.835514),
 ('mixUp',): array(85238.835514),
 ('mixDown',): array(85238.